# Benchmark: CPU vs MPS (Apple GPU)

Times the prior of the paper (`SO3PriorTrainer` with the setup and weights of `id_0900`) on each available device:

- **train step:** forward, backward, gradient clipping, optimizer and scheduler step (the real `training_step`), as in the training;
- **log-likelihood:** forward pass without gradients, as in the evaluation;
- **sampling:** generating poses (inverse pass of the flow).

Each is timed for the batch sizes in `BATCH_SIZES`, after a few warm-up runs, with the device synchronized before reading the clock. The inputs are fixed random rotations; the time does not depend on their values (except slightly for sampling).

It also checks that **MPS gives the same log-likelihoods as CPU** (same weights, same inputs) and lists operators that MPS does not support and that **fall back to the CPU** (each fallback copies data between GPU and CPU and is slow).

Run it when nothing else is training: other notebooks on the CPU distort the CPU times. Restart the kernel before running: the MPS fallback setting below must be set before torch is imported.

In [ ]:
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")  # must be set before torch is imported (restart the kernel)

import importlib
import json
import time
import warnings
from pathlib import Path

import numpy as np
import roma
import torch
from omegaconf import OmegaConf

from hp.utils.paths import REPO_ROOT

# ---- parameters ----------------------------------------------------------------------
PRIOR = REPO_ROOT / "experiments" / "pretrained_models" / "id_0900"  # setup and weights
BATCH_SIZES = [500, 2000]     # 2000 = training batch of id_0900
TRAIN_STEPS = 10              # timed steps per batch size (after WARMUP)
EVAL_RUNS = 10
SAMPLE_RUNS = 3
WARMUP = 3
OUT_DIR = "~/data/experiments/benchmark_device"
QUICK = False                 # True: tiny run to check that the notebook works
# ----------------------------------------------------------------------------------------

if QUICK:
    BATCH_SIZES, TRAIN_STEPS, EVAL_RUNS, SAMPLE_RUNS, WARMUP = [16], 2, 2, 1, 1
DEVICES = ["cpu"] + (["mps"] if torch.backends.mps.is_available() else [])
OUT_DIR = Path(OUT_DIR).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"torch {torch.__version__} | devices: {DEVICES} | CPU threads: {torch.get_num_threads()} | "
      f"MPS fallback: {os.environ['PYTORCH_ENABLE_MPS_FALLBACK']}")

## Helpers

In [ ]:
def sync(device):
    if device == "mps":
        torch.mps.synchronize()


def make_trainer(device):
    """SO3PriorTrainer of the prior on the device, with its weights, its optimizer and scheduler; no data loaded."""
    os.environ["HUPROSO_DEVICE"] = device  # the device setting wins over config.device
    cfg = OmegaConf.load(PRIOR / "config.yaml")
    cfg.only_eval, cfg.evaluate, cfg.standard_mode, cfg.condition = True, False, True, False
    cfg.device, cfg.save.path_dir = device, str(PRIOR)
    cfg.clip_grad_norm = 1.0  # as in the training (not in the old config)
    torch.manual_seed(0)
    trainer = getattr(importlib.import_module("hp.core.trainer"), cfg.trainer)(cfg)
    trainer.load_model(save_path=str(PRIOR), device=device)
    trainer.train_dataset = range(10**6)  # only its length is read by the scheduler
    trainer.create_optimizer_and_scheduler()
    assert str(next(trainer.pdf_estimator.parameters()).device).startswith(device)
    return trainer


def timed(device, fn, runs, warmup=WARMUP):
    """Seconds per call of fn: the median of `runs` calls after `warmup` calls."""
    for _ in range(warmup):
        fn()
    sync(device)
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        fn()
        sync(device)
        times.append(time.perf_counter() - start)
    return float(np.median(times))


def rotations(batch, joints=19, seed=0):
    torch.manual_seed(seed)
    return roma.random_rotmat(batch * joints).reshape(batch, joints, 3, 3).float()

## Run

Per device: the log-likelihood of fixed inputs (for the comparison between devices), then the timings. An operation that fails on a device is reported, not raised.

In [ ]:
results, fallbacks, reference_ll = {}, {}, {}
for device in DEVICES:
    print(f"--- {device}")
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        trainer = make_trainer(device)
        trainer.pdf_estimator.eval()
        with torch.no_grad():
            reference_ll[device] = trainer.forward_estimator(rotations(256).to(device)).cpu()
        for batch in BATCH_SIZES:
            x = rotations(batch).to(device)
            row = {}

            def eval_ll():
                with torch.no_grad():
                    trainer.forward_estimator(x)

            def sample():
                with torch.no_grad():
                    trainer.sample_from_estimator(batch)

            def train_step():
                trainer.training_step(x)

            for name, fn, runs in (("log_likelihood", eval_ll, EVAL_RUNS), ("sampling", sample, SAMPLE_RUNS),
                                   ("train_step", train_step, TRAIN_STEPS)):
                if name == "train_step":
                    trainer.pdf_estimator.train()
                try:
                    row[name] = timed(device, fn, runs)
                    print(f"batch {batch:5d} | {name:15s} {1000 * row[name]:9.1f} ms")
                except Exception as e:  # e.g. an operation not available on MPS
                    row[name] = None
                    print(f"batch {batch:5d} | {name:15s} FAILED: {type(e).__name__}: {str(e)[:200]}")
            results[(device, batch)] = row
    fallbacks[device] = sorted({str(w.message).split("'")[1] for w in caught
                                if "fall back to run on the CPU" in str(w.message) and "'" in str(w.message)})
    if fallbacks[device]:
        print(f"operators that fell back to the CPU on {device}: {', '.join(fallbacks[device])}")

## Result

Time per call (median) and the speed-up of MPS over CPU (above 1: MPS is faster).

In [ ]:
print(f"{'batch':>6s} {'operation':15s} " + " ".join(f"{d:>10s}" for d in DEVICES) + ("   MPS speed-up" if "mps" in DEVICES else ""))
for batch in BATCH_SIZES:
    for name in ("train_step", "log_likelihood", "sampling"):
        times = [results[(d, batch)].get(name) for d in DEVICES]
        line = f"{batch:6d} {name:15s} " + " ".join(f"{1000 * t:8.1f}ms" if t else f"{'failed':>10s}" for t in times)
        if "mps" in DEVICES and all(times):
            line += f"   {times[0] / times[1]:6.2f}x"
        print(line)

if "mps" in DEVICES:
    diff = (reference_ll["mps"] - reference_ll["cpu"]).abs()
    print(f"\nlog-likelihood MPS vs CPU on 256 poses: max difference {diff.max():.2e}, mean {diff.mean():.2e} "
          f"(values around {reference_ll['cpu'].abs().mean():.0f})")
    print("operators that fell back to the CPU:", ", ".join(fallbacks["mps"]) or "none")
    t = results[("cpu", 2000)].get("train_step") if 2000 in BATCH_SIZES else None
    m = results[("mps", 2000)].get("train_step") if 2000 in BATCH_SIZES else None
    if t and m:
        print(f"\n2000 training iterations at batch 2000: CPU {2000 * t / 3600:.1f} h, MPS {2000 * m / 3600:.1f} h")

with open(OUT_DIR / "benchmark_device.json", "w") as f:
    json.dump(dict(torch=torch.__version__, threads=torch.get_num_threads(), fallbacks=fallbacks,
                   results=[dict(device=d, batch=b, **r) for (d, b), r in results.items()]), f, indent=1)
print("saved to", OUT_DIR)